# Trivial Zebra Puzzle — AC-3 Solver

We model each color and animal as a variable whose value is the **house position** (1–3).
AC-3 enforces arc consistency across the binary constraints to narrow domains.


In [ ]:
from collections import deque

HOUSES = {1, 2, 3}

variables = [
    'Red', 'Blue', 'Green',
    'Cat', 'Dog', 'Zebra'
]

# All variables start with all possible houses
domains = {var: set(HOUSES) for var in variables}

def neq(x, y):
    return x != y

def eq(x, y):
    return x == y

def left_of(x, y):
    return x == y - 1

def is_middle(x):
    return x == 2

constraints = {}

def add_binary_constraint(xi, xj, predicate):
    constraints.setdefault((xi, xj), []).append(predicate)

# All-different constraints (Standard CSP requirement)
color_vars = ['Red', 'Blue', 'Green']
for i, xi in enumerate(color_vars):
    for xj in color_vars[i + 1:] :
        add_binary_constraint(xi, xj, neq)
        add_binary_constraint(xj, xi, neq)

animal_vars = ['Cat', 'Dog', 'Zebra']
for i, xi in enumerate(animal_vars):
    for xj in animal_vars[i + 1:] :
        add_binary_constraint(xi, xj, neq)
        add_binary_constraint(xj, xi, neq)

# Clues as formal constraints
add_binary_constraint('Cat', 'Red', eq)
add_binary_constraint('Red', 'Cat', eq)
add_binary_constraint('Blue', 'Red', left_of)
add_binary_constraint('Dog', 'Blue', eq)
add_binary_constraint('Blue', 'Dog', eq)

# Unary constraint for "Red is middle"
# In a pure AC-3, we model this as a self-arc or a filtered initial domain
def apply_unary_constraints(domains):
    domains['Red'] = {x for x in domains['Red'] if is_middle(x)}

def revise(domains, xi, xj):
    revised = False
    predicates = constraints.get((xi, xj), [])
    if not predicates: return revised

    to_remove = set()
    for x in domains[xi]:
        if not any(all(pred(x, y) for pred in predicates) for y in domains[xj]):
            to_remove.add(x)

    if to_remove:
        domains[xi] -= to_remove
        revised = True
    return revised

def ac3(domains):
    # Apply unary constraints first to establish the starting state
    apply_unary_constraints(domains)
    
    queue = deque(constraints.keys())
    while queue:
        xi, xj = queue.popleft()
        if revise(domains, xi, xj):
            if not domains[xi]: return False
            for xk in variables:
                if xk != xi and (xk, xi) in constraints:
                    queue.append((xk, xi))
    return True

ac3(domains)
domains

In [ ]:
zebra_house = next(iter(domains['Zebra']))
f'Zebra lives in house {zebra_house}.'
